# The Availability Gap — reproducible analysis
Independent portfolio project. Morrowfield Food Co. and the dataset are fictional.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
root = Path.cwd()
if not (root / 'data' / 'clean').exists():
    root = root.parent
connection = sqlite3.connect(root / 'data' / 'clean' / 'morrowfield_analytics.sqlite')


In [ ]:
pd.read_sql_query('''
SELECT SUM(NetSales) AS net_sales, SUM(GrossProfit) AS gross_profit,
       SUM(GrossProfit) / SUM(NetSales) AS gross_margin
FROM FactSalesWeekly
''', connection)


In [ ]:
pd.read_sql_query('''
SELECT s.SupplierName, SUM(i.EstimatedLostSales) AS estimated_lost_sales,
       SUM(i.EstimatedLostSales) / SUM(SUM(i.EstimatedLostSales)) OVER () AS loss_share
FROM FactInventoryWeekly i JOIN DimSuppliers s ON s.SupplierID = i.SupplierID
GROUP BY s.SupplierName ORDER BY estimated_lost_sales DESC
''', connection).head(8)


In [ ]:
pd.read_sql_query('''
SELECT ds.Region, SUM(fs.NetSales) AS net_sales,
       SUM(fs.GrossProfit) / SUM(fs.NetSales) AS gross_margin,
       SUM(fs.DiscountAmount) / SUM(fs.GrossSales) AS discount_rate
FROM FactSalesWeekly fs JOIN DimStores ds ON ds.StoreID = fs.StoreID
GROUP BY ds.Region ORDER BY net_sales DESC
''', connection)


## Interpretation
Supplier and regional differences describe associations. Estimated lost sales depend on the documented category-specific demand-capture assumptions; they are not recorded revenue.
